In [ ]:
# ==============================================================================
# File: 02_Train_ZeroFake.ipynb (Cell 1 of 7)
# What this notebook does: Mounts Google Drive, clones/pulls https://github.com/Alekhagrawal/ZeroFake.git,
#   unzips cropped_faces.zip onto Colab's fast local SSD (/content/data), and runs all four
#   ZeroFake ablation training experiments (Baseline, Contrastive, Single-Prototype, and Full ZeroFake)
#   on an L4 or A100 GPU with fp16 mixed precision and per-epoch Google Drive checkpointing.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/cropped_faces.zip and GitHub repo.
# Outputs: Saved last_checkpoint.pt and best_model.pt (including calibrated UNKNOWN
#   threshold tau_reject) inside /content/drive/MyDrive/ZeroFake_Storage/checkpoints/.
# ==============================================================================

# --- CELL 1: Mount Google Drive & Verify cropped_faces.zip Exists ---
import os
from google.colab import drive

# 1. Mount Google Drive at /content/drive
drive.mount('/content/drive')

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')

# 2. Verify that cropped_faces.zip from 01_Prepare_Data.ipynb exists
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'cropped_faces.zip not found at {ZIP_PATH}! '
        'Please run 01_Prepare_Data.ipynb first to prepare and back up the dataset.'
    )

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'[SUCCESS] Found cropped_faces.zip on Google Drive ({zip_size_mb:.2f} MB): {ZIP_PATH}')


In [ ]:
# --- CELL 2: Clone or Pull Latest Code from GitHub & Install Dependencies ---
# Plain-English explanation:
#   Clones https://github.com/Alekhagrawal/ZeroFake.git into /content/ZeroFake (or pulls
#   the latest commits if already cloned), switches the working directory to /content/ZeroFake,
#   installs facenet-pytorch with --no-deps first (avoiding Pillow 10.2.0 compilation on Python 3.13),
#   and installs the remaining packages using 'grad-cam'.

import os

GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFake.git'
REPO_DIR = '/content/ZeroFake'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Pulling latest updates from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFake pull
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !git clone https://github.com/Alekhagrawal/ZeroFake.git /content/ZeroFake

%cd /content/ZeroFake
!ls -la

# 1. Install facenet-pytorch separately with --no-deps so pip does not compile Pillow 10.2.0 on Python 3.13
!pip install --no-deps facenet-pytorch

# 2. Install remaining packages using the official PyPI package name 'grad-cam'
!pip install timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio pyyaml tqdm

print('[SUCCESS] Repository ready at /content/ZeroFake and all dependencies installed.')


In [ ]:
# --- CELL 3: Unzip cropped_faces.zip onto Colab's Fast Local SSD (/content/data) ---
# Plain-English explanation:
#   Reading 60,000 individual image files directly across Google Drive causes severe
#   network latency. Unzipping cropped_faces.zip once onto Colab's local /content/data
#   SSD lets the L4/A100 GPU train at maximum speed (~20-35 minutes per 15-epoch run).

import os
import shutil
import zipfile

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

manifest_local = os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')
if not os.path.exists(manifest_local):
    print(f'[INFO] Unzipping {ZIP_PATH} onto local SSD {LOCAL_DATA_DIR}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    print('[SUCCESS] Extracted all cropped faces onto local SSD (/content/data).')
else:
    print('[INFO] Cropped faces already extracted in /content/data.')

# Also copy splits.json and frames_manifest.csv if stored alongside cropped_faces.zip on Drive
for fname in ('splits.json', 'frames_manifest.csv'):
    drive_file = os.path.join(STORAGE_DIR, fname)
    local_file = os.path.join(LOCAL_DATA_DIR, fname)
    if os.path.exists(drive_file) and not os.path.exists(local_file):
        shutil.copy2(drive_file, local_file)

print(f'[READY] Local SSD dataset directory contents: {os.listdir(LOCAL_DATA_DIR)}')


In [ ]:
# --- CELL 4: Train Config A (Baseline: --mode baseline) for 15 Epochs ---
# Plain-English explanation:
#   Trains the standard ConvNeXt-Tiny binary classifier baseline (with no contrastive
#   loss or prototypes) for 15 epochs. Saves last_checkpoint.pt and best_model.pt
#   after every epoch to /content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline/
#   and automatically resumes if a previous run was interrupted.

import os

%cd /content/ZeroFake
BASELINE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline'
os.makedirs(BASELINE_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 1 — Config A (Baseline: --mode baseline) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode baseline --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/baseline --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config A (Baseline) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config A (Baseline) checkpoints saved to: {BASELINE_SAVE_DIR}')


In [ ]:
# --- CELL 5: Train Config B (Contrastive: --mode contrastive) for 15 Epochs ---
# Plain-English explanation:
#   Trains Run 2 (Config B — contrastive), adding Multi-Generator Supervised Contrastive
#   Loss (L_con) to align fake features across generators and cluster real faces tightly.
#   Saves checkpoints after every epoch to /content/drive/MyDrive/ZeroFake_Storage/checkpoints/contrastive/.

import os

%cd /content/ZeroFake
CONTRASTIVE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/contrastive'
os.makedirs(CONTRASTIVE_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 2 — Config B (Contrastive: --mode contrastive) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode contrastive --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/contrastive --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config B (Contrastive) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config B (Contrastive) checkpoints saved to: {CONTRASTIVE_SAVE_DIR}')


In [ ]:
# --- CELL 6: Train Config C (Single Prototype: --mode single_proto) for 15 Epochs ---
# Plain-English explanation:
#   Trains Run 3 (Config C — single_proto), adding 1 Real prototype and 1 Fake prototype (K=1)
#   to test the single-cluster fake assumption before upgrading to K=6 diverse Fake prototypes.
#   Saves checkpoints after every epoch to /content/drive/MyDrive/ZeroFake_Storage/checkpoints/single_proto/.

import os

%cd /content/ZeroFake
SINGLE_PROTO_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/single_proto'
os.makedirs(SINGLE_PROTO_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 3 — Config C (Single Prototype: --mode single_proto) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode single_proto --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/single_proto --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Config C (Single Prototype) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Config C (Single Prototype) checkpoints saved to: {SINGLE_PROTO_SAVE_DIR}')


In [ ]:
# --- CELL 7: Train Full ZeroFake (Config D/E: --mode zerofake) for 15 Epochs ---
# Plain-English explanation:
#   Trains Run 4 (Config D/E — Full ZeroFake) combining ConvNeXt-Tiny, Multi-Generator
#   Supervised Contrastive Learning (L_con), 1 Real + K=6 diverse Fake prototypes (L_proto),
#   and validation-calibrated Open-Set Uncertainty U(x). Saves best_model.pt (with the
#   calibrated UNKNOWN threshold tau_reject locked in at 95% validation coverage) to
#   /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/.

import os

%cd /content/ZeroFake
ZEROFAKE_SAVE_DIR = '/content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake'
os.makedirs(ZEROFAKE_SAVE_DIR, exist_ok=True)

print('[TRAIN] Starting Run 4 — Config D/E (Full ZeroFake: --mode zerofake) for 15 epochs...')
!PYTHONPATH=/content/ZeroFake python -u training/train.py --mode zerofake --epochs 15 --batch_size 64 --lr 1e-4 --data_dir /content/data --save_dir /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake --resume auto
if _exit_code != 0:
    raise RuntimeError(f'Full ZeroFake (Config D/E) training failed with exit code {_exit_code}')
print(f'[SUCCESS] Full ZeroFake best_model.pt (with calibrated UNKNOWN threshold) saved to: {ZEROFAKE_SAVE_DIR}')
